In [ ]:
# Core Python
import os
import math
import random
import time
import copy
from pathlib import Path
from collections import Counter, defaultdict

# Data Handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress / Utilities
from tqdm import tqdm

# Scikit-learn (optional utilities for splits / metrics)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# PyTorch Core
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# PyTorch Data Utilities
from torch.utils.data import Dataset, DataLoader

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [1]:
from collections import Counter
import numpy as np
import pandas as pd

# Load cleaned corpus (one document per line)
with open("cleaned.txt", "r", encoding="utf-8") as f:
    documents = [line.strip() for line in f if line.strip()]

# Tokenize documents
tokenized_docs = [doc.split() for doc in documents]

# Count token frequencies
token_counts = Counter()
for doc in tokenized_docs:
    token_counts.update(doc)

# Keep top 10,000 most frequent tokens
MAX_VOCAB = 10000
most_common_tokens = token_counts.most_common(MAX_VOCAB)

# Build vocabulary
vocab = {"<UNK>": 0}
for idx, (token, _) in enumerate(most_common_tokens, start=1):
    vocab[token] = idx

# Reverse lookup (optional)
idx_to_token = {idx: token for token, idx in vocab.items()}

# Build term-document matrix
num_docs = len(tokenized_docs)
vocab_size = len(vocab)

td_matrix = np.zeros((num_docs, vocab_size), dtype=np.int32)

for doc_idx, doc in enumerate(tokenized_docs):
    for token in doc:
        token_id = vocab.get(token, 0)   # map rare/unseen to <UNK>
        td_matrix[doc_idx, token_id] += 1

# Convert to DataFrame (optional)
td_df = pd.DataFrame(td_matrix, columns=[idx_to_token[i] for i in range(vocab_size)])

print("Documents:", num_docs)
print("Vocabulary size:", vocab_size)
print("Term-document matrix shape:", td_matrix.shape)

# Preview
td_df.head()

Documents: 15803
Vocabulary size: 10001
Term-document matrix shape: (15803, 10001)


,<UNK>,کے,میں,کی,اور,سے,کہ,کا,نے,کو,...,پرجھکاؤ,[6],سیٹلائیٹ,ٹاؤن،,سکستھ,روڈ،,ایموجی,خریدوں,لگاؤں,ایکسپو
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,3,3,4,0,1,1,0,0,2,...,0,0,0,0,0,0,0,0,0,0
3,0,2,2,3,1,2,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,2,2,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import numpy as np
import pandas as pd

# --- Inputs expected from previous cell ---
# td_matrix        -> term-document count matrix (num_docs x vocab_size)
# vocab            -> token -> index
# idx_to_token     -> index -> token
# documents        -> raw docs list
# tokenized_docs   -> tokenized docs list

# ---------------------------------------------------------
# If you have labels in a separate file (recommended):
# one topic/category label per line aligned with cleaned.txt
# Example filename: labels.txt
# ---------------------------------------------------------
with open("labels.txt", "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

assert len(labels) == td_matrix.shape[0], "labels.txt rows must match cleaned.txt docs"

# ===============================
# Compute TF-IDF
# TF-IDF(w,d) = TF(w,d) * log(N / (1 + df(w)))
# ===============================
N = td_matrix.shape[0]

# document frequency: number of docs containing term
df = np.sum(td_matrix > 0, axis=0)   # shape: (vocab_size,)

# idf
idf = np.log(N / (1 + df))

# tf-idf matrix
tfidf_matrix = td_matrix * idf

# Save matrix
np.save("tfidf_matrix.npy", tfidf_matrix)

print("Saved: tfidf_matrix.npy")
print("Shape:", tfidf_matrix.shape)

# ===============================
# Top-10 discriminative words per topic
# Method:
# Average TF-IDF within topic
# ===============================
unique_topics = sorted(set(labels))

for topic in unique_topics:
    doc_ids = [i for i, y in enumerate(labels) if y == topic]

    topic_scores = tfidf_matrix[doc_ids].mean(axis=0)   # avg score per word

    top_ids = np.argsort(topic_scores)[::-1][:10]

    top_words = [(idx_to_token[i], round(float(topic_scores[i]), 4)) for i in top_ids]

    print(f"\nTopic: {topic}")
    print("-" * 40)
    for rank, (word, score) in enumerate(top_words, 1):
        print(f"{rank:2d}. {word:20s} {score}")